# GAFormer — Sinh artifact còn thiếu cho luận văn

Notebook này sinh **tất cả** file còn thiếu sau khi đã train xong:

| Bước | Task | Output |
|------|------|--------|
| 1 | Kiểm tra GPU & Mount Drive | — |
| 2 | Upload 2 script mới lên Drive (nếu chưa có) | — |
| 3+4 | Confusion matrix + t-SNE cho 4 model so sánh | `*_confusion.png` + `*_tsne.png` × 4 |
| 5 | Per-class report GAFormer chính | `per_class_report.csv`, `per_class_bar.png` |
| 6 | Hiển thị tất cả ảnh inline | — |
| 7 | Zip & tải về máy | `gaformer_artifacts.zip` |

> **Yêu cầu:** Vào `Runtime → Change runtime type → T4 GPU` trước khi chạy.

## Bước 1 — Kiểm tra GPU & Mount Drive

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU : {gpu}  |  VRAM: {mem:.1f} GB")
else:
    print("Khong co GPU — vao Runtime > Change runtime type > T4 GPU")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path

# ── Đường dẫn — chỉnh nếu cấu trúc Drive khác ───────────────────────────
SRC_DIR      = Path('/content/drive/MyDrive/DOAN2/gaformer/src')
CKPT_DIR     = Path('/content/drive/MyDrive/DOAN2/gaformer/checkpoints')
COMP_DIR     = CKPT_DIR / 'comparison'
RUN_DIR      = CKPT_DIR / 'gaformer_20260525_113523'
MANIFEST     = Path('/content/drive/MyDrive/DOAN2/SHG_Dataset/manifest_shg.csv')
DATASET_ROOT = Path('/content/drive/MyDrive/DOAN2/SHG_Dataset')
LABELS_JSON  = RUN_DIR / 'labels.json'
# ─────────────────────────────────────────────────────────────────────────

print(f"SRC_DIR  : {SRC_DIR}  -> exists={SRC_DIR.exists()}")
print(f"COMP_DIR : {COMP_DIR}  -> exists={COMP_DIR.exists()}")
print(f"RUN_DIR  : {RUN_DIR}  -> exists={RUN_DIR.exists()}")
print(f"MANIFEST : {MANIFEST}  -> exists={MANIFEST.exists()}")

## Bước 2 — Upload 2 script mới vào Drive (chỉ làm 1 lần)

Kiểm tra xem `compare_confusion.py` và `generate_report.py` đã có chưa.  
Nếu thiếu → chạy cell upload bên dưới, chọn 2 file đó từ máy local.

In [ ]:
NEEDED = ['compare_confusion.py', 'generate_report.py']
missing = [f for f in NEEDED if not (SRC_DIR / f).exists()]

if missing:
    print(f"THIEU: {missing}")
    print("=> Chay cell upload phia duoi de upload 2 file do.")
else:
    print("Da co du 2 script — bo qua buoc upload.")
    print(f"Files trong src/: {sorted(f.name for f in SRC_DIR.glob('*.py'))}")

In [ ]:
# Chỉ chạy nếu cell trên báo THIEU
# Chọn file: compare_confusion.py và generate_report.py
from google.colab import files

print("Chon 2 file: compare_confusion.py va generate_report.py")
uploaded = files.upload()

SRC_DIR.mkdir(parents=True, exist_ok=True)
for fname, content in uploaded.items():
    dest = SRC_DIR / fname
    dest.write_bytes(content)
    print(f"Da luu: {dest}")

## Bước 3 & 4 — Confusion matrix + t-SNE cho 4 model so sánh

Sinh 8 file vào `checkpoints/comparison/`:

```
gadf_coatnet0_confusion.png   gadf_coatnet0_tsne.png
gasf_coatnet0_confusion.png   gasf_coatnet0_tsne.png
gadf_resnet50_confusion.png   gadf_resnet50_tsne.png
gaformer_confusion.png        gaformer_tsne.png
```

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(SRC_DIR / 'compare_confusion.py'),
    '--comp-dir',   str(COMP_DIR),
    '--labels',     str(LABELS_JSON),
    '--manifest',   str(MANIFEST),
    '--root',       str(DATASET_ROOT),
    '--batch-size', '32',
]

print('Chay:', ' '.join(cmd))
print('-' * 60)
result = subprocess.run(cmd, cwd=str(SRC_DIR))

print('-' * 60)
if result.returncode != 0:
    print('LOI — xem output phia tren de debug')
else:
    generated = sorted(COMP_DIR.glob('*_confusion.png')) + sorted(COMP_DIR.glob('*_tsne.png'))
    print(f'Hoan tat! {len(generated)} file sinh ra:')
    for f in generated:
        print(f'  {f.name}')

## Bước 5 — Per-class classification report cho GAFormer chính

Sinh 3 file vào `checkpoints/gaformer_20260525_113523/`:
- `per_class_report.csv` — bảng Precision / Recall / F1 của 20 class
- `per_class_report.json`
- `per_class_bar.png` — biểu đồ bar chart F1 từng class

In [ ]:
cmd2 = [
    sys.executable, str(SRC_DIR / 'generate_report.py'),
    '--run-dir',  str(RUN_DIR),
    '--manifest', str(MANIFEST),
    '--root',     str(DATASET_ROOT),
]

print('Chay:', ' '.join(cmd2))
print('-' * 60)
result2 = subprocess.run(cmd2, cwd=str(SRC_DIR))

print('-' * 60)
if result2.returncode != 0:
    print('LOI — xem output phia tren de debug')
else:
    print('Hoan tat! Files sinh ra:')
    for fname in ['per_class_report.csv', 'per_class_report.json', 'per_class_bar.png']:
        p = RUN_DIR / fname
        print(f'  {fname}  ->  exists={p.exists()}')

## Bước 6 — Hiển thị tất cả ảnh vừa sinh

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show(path, title=None):
    p = Path(path)
    if not p.exists():
        print(f"  Chua co: {p.name}")
        return
    img = mpimg.imread(str(p))
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title or p.name, fontsize=12)
    plt.tight_layout()
    plt.show()

METHODS = [
    ('gadf_coatnet0', 'GADF + CoAtNet-0'),
    ('gasf_coatnet0', 'GASF + CoAtNet-0'),
    ('gadf_resnet50', 'GADF + ResNet-50'),
    ('gaformer',      'GAFormer (de xuat)'),
]

print('=== Confusion Matrix ===')
for method, label in METHODS:
    show(COMP_DIR / f'{method}_confusion.png', f'Confusion Matrix — {label}')

print('\n=== t-SNE ===')
for method, label in METHODS:
    show(COMP_DIR / f'{method}_tsne.png', f't-SNE — {label}')

print('\n=== Per-class F1 bar chart ===')
show(RUN_DIR / 'per_class_bar.png', 'F1 tung class — GAFormer')

In [ ]:
# Xem bảng per-class report dạng text
import pandas as pd

csv_path = RUN_DIR / 'per_class_report.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    pd.set_option('display.max_rows', 30)
    display(df)
else:
    print(f"Chua co {csv_path.name} — chay lai Buoc 5")

## Bước 7 — Zip tất cả artifact & tải về máy

In [ ]:
import shutil
from datetime import datetime
from google.colab import files

ts      = datetime.now().strftime('%Y%m%d_%H%M')
tmp_dir = Path(f'/content/artifacts_tmp_{ts}')
tmp_dir.mkdir()

# --- Comparison: confusion + t-SNE 4 model ---
comp_out = tmp_dir / 'comparison'
comp_out.mkdir()
for pattern in ['*_confusion.png', '*_tsne.png']:
    for f in COMP_DIR.glob(pattern):
        shutil.copy(f, comp_out / f.name)

# --- GAFormer chính ---
main_out = tmp_dir / 'gaformer_main'
main_out.mkdir()
for fname in [
    'training_curve.png', 'confusion_matrix.png', 'tsne.png',
    'per_class_bar.png', 'per_class_report.csv',
    'per_class_report.json', 'test_metrics.json',
]:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy(src, main_out / fname)

# --- Tạo zip ---
zip_path = f'/content/gaformer_artifacts_{ts}'
shutil.make_archive(zip_path, 'zip', str(tmp_dir))
shutil.rmtree(tmp_dir)

size_kb = Path(f'{zip_path}.zip').stat().st_size / 1024
print(f"Da tao: gaformer_artifacts_{ts}.zip  ({size_kb:.0f} KB)")
print("Noi dung:")
for label, folder in [('comparison/', comp_out), ('gaformer_main/', main_out)]:
    n = len(list(folder.glob('*'))) if folder.exists() else 0
    print(f"  {label}: {n} files")

files.download(f'{zip_path}.zip')
print("Dang tai ve may...")